In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib widget

In [ ]:
FILENAME = "file.csv"

data = np.loadtxt(FILENAME, delimiter=",", skiprows=1)

times = data[:, 0]
currents_msp = data[:, 1]
currents_rx = data[:, 2]
currents_bat = data[:, 3]

In [ ]:
AVG_WINDOW_SIZE = 1000
THRESHOLD_CURRENT_PERCENT = 50

times_smooth = times[AVG_WINDOW_SIZE - 1:]
currents_msp_smooth = np.convolve(currents_msp, np.ones(AVG_WINDOW_SIZE) / AVG_WINDOW_SIZE, mode='valid')
currents_rx_smooth = np.convolve(currents_rx, np.ones(AVG_WINDOW_SIZE) / AVG_WINDOW_SIZE, mode='valid')
currents_bat_smooth = np.convolve(currents_bat, np.ones(AVG_WINDOW_SIZE) / AVG_WINDOW_SIZE, mode='valid')

currents_msp_avg = np.mean(currents_msp)
currents_rx_avg = np.mean(currents_rx)
currents_bat_avg = np.mean(currents_bat)

# Start index: first index where smooth battery current is above threshold
idx_start = next(i for i, v in enumerate(currents_bat_smooth) if v > (THRESHOLD_CURRENT_PERCENT / 100) * currents_bat_avg)
# End index: last index where smooth battery current is above threshold
idx_end = len(currents_bat_smooth) - next(i for i, v in enumerate(reversed(currents_bat_smooth)) if v > (THRESHOLD_CURRENT_PERCENT / 100) * currents_bat_avg)

print(f"Measurement Region: {times_smooth[idx_start]:.2f}s to {times_smooth[idx_end-1]:.2f}s")

In [ ]:
avg_current_msp_a = np.mean(currents_msp[idx_start:idx_end])
avg_current_rx_a = np.mean(currents_rx[idx_start:idx_end])
avg_current_bat_a = np.mean(currents_bat[idx_start:idx_end])

avg_power_msp_mw = avg_current_msp_a * 3.3 * 1000
avg_power_rx_mw = avg_current_rx_a * 3.3 * 1000
avg_power_bat_mw = avg_current_bat_a * 3.7 * 1000

print(f"Average MSP430 Power: {avg_power_msp_mw:.2f} mW")
print(f"Average RX Power: {avg_power_rx_mw:.2f} mW")
print(f"Average Battery Power: {avg_power_bat_mw:.2f} mW")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), ncols=2, sharey=True)

ax[0].plot(times, currents_msp * 1000, label="MSP Current (mA)", alpha=0.3)
ax[0].plot(times, currents_rx * 1000, label="RX Current (mA)", alpha=0.3)
ax[0].plot(times, currents_bat * 1000, label="Battery Current (mA)", alpha=0.3)
ax[0].plot(times_smooth, currents_msp_smooth * 1000, label="MSP Current (mA) - Running Avg", color='blue')
ax[0].plot(times_smooth, currents_rx_smooth * 1000, label="RX Current (mA) - Running Avg", color='orange')
ax[0].plot(times_smooth, currents_bat_smooth * 1000, label="Battery Current (mA) - Running Avg", color='green')
ax[0].axhline(y=avg_current_msp_a * 1000, color='blue', linestyle='--', label=f"Avg MSP Current: {avg_current_msp_a*1000:.2f} mA") # type: ignore
ax[0].axhline(y=avg_current_rx_a * 1000, color='orange', linestyle='--', label=f"Avg RX Current: {avg_current_rx_a*1000:.2f} mA") # type: ignore
ax[0].axhline(y=avg_current_bat_a * 1000, color='green', linestyle='--', label=f"Avg Battery Current: {avg_current_bat_a*1000:.2f} mA") # type: ignore
ax[0].axvline(x=times_smooth[idx_start], color='red', linestyle=':', label="Measurement Start/End")
ax[0].set_xlabel("Time [s]")
ax[0].set_ylabel("Current [mA]")
ax[0].set_xlim(times_smooth[idx_start] - 0.1, times_smooth[idx_start] + 0.5)
ax[0].set_ylim(-0.05e3, 0.4e3)
ax[0].grid(True)

ax[1].plot(times, currents_msp * 1000, label="MSP Current (mA)", alpha=0.3)
ax[1].plot(times, currents_rx * 1000, label="RX Current (mA)", alpha=0.3)
ax[1].plot(times, currents_bat * 1000, label="Battery Current (mA)", alpha=0.3)
ax[1].plot(times_smooth, currents_msp_smooth * 1000, label="MSP Current (mA) - Running Avg", color='blue')
ax[1].plot(times_smooth, currents_rx_smooth * 1000, label="RX Current (mA) - Running Avg", color='orange')
ax[1].plot(times_smooth, currents_bat_smooth * 1000, label="Battery Current (mA) - Running Avg", color='green')
ax[1].axhline(y=avg_current_msp_a * 1000, color='blue', linestyle='--', label=f"Avg MSP Current: {avg_current_msp_a*1000:.2f} mA") # type: ignore
ax[1].axhline(y=avg_current_rx_a * 1000, color='orange', linestyle='--', label=f"Avg RX Current: {avg_current_rx_a*1000:.2f} mA") # type: ignore
ax[1].axhline(y=avg_current_bat_a * 1000, color='green', linestyle='--', label=f"Avg Battery Current: {avg_current_bat_a*1000:.2f} mA") # type: ignore
ax[1].axvline(x=times_smooth[idx_end], color='red', linestyle=':', label="Measurement Start/End")
ax[1].set_xlabel("Time [s]")
ax[1].set_xlim(times_smooth[idx_end] - 0.5, times_smooth[idx_end] + 0.1)
ax[1].set_ylim(-0.05e3, 0.4e3)
ax[1].grid(True)
ax[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax1.semilogy(times, currents_msp * 1e3, label="MSP Current [mA]", color='black', alpha=0.7)
ax1.axhline(y=avg_current_msp_a * 1e3, color='red', linestyle='--', label=f"Avg: {avg_current_msp_a*1e3:.2f} mA") # type: ignore
ax1.set_ylabel("Current (mA)")
ax1.set_title("MSP430 Current Consumption")
ax1.legend()
ax1.grid(True)
ax2.semilogy(times, currents_rx * 1e3, label="RX Current [mA]", color='black', alpha=0.7)
ax2.axhline(y=avg_current_rx_a * 1e3, color='red', linestyle='--', label=f"Avg: {avg_current_rx_a*1e3:.2f} mA") # type: ignore
ax2.set_ylabel("Current (mA)")
ax2.set_title("RX Current Consumption")
ax2.legend()
ax2.grid(True)
ax3.semilogy(times, currents_bat * 1e3, label="Battery Current [mA]", color='black', alpha=0.7)
ax3.axhline(y=avg_current_bat_a * 1e3, color='red', linestyle='--', label=f"Avg: {avg_current_bat_a*1e3:.2f} mA") # type: ignore
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("Current (mA)")
ax3.set_title("Battery Current Consumption")
ax3.legend()
ax3.grid(True)
plt.xlim((times_smooth[idx_end] - times_smooth[idx_start]) / 2 - 0.2, (times_smooth[idx_end] - times_smooth[idx_start]) / 2 - 0.1 + 0.2)
plt.tight_layout()
plt.show()